# Dim_Particular - Gold

Unique category and particular pairs from silver regional appropriations and obligations tables

## 1. Create the dimension table

- Combine category–particular pairs from both Silver tables.
- Trim spaces and convert text to uppercase.
- Exclude blank categories, blank particulars, and `ALL` particulars.
- Remove duplicate pairs.
- Assign `particular_id` values starting at `30000`.


In [0]:
%sql
CREATE TABLE IF NOT EXISTS pondolens.dev_gold.dim_particular
USING DELTA
AS
WITH distinct_particulars AS (
  SELECT
    UPPER(TRIM(CAST(category AS STRING))) AS category,
    UPPER(TRIM(CAST(particular AS STRING))) AS particular
  FROM pondolens.dev_silver.regional_appropriations
  WHERE NULLIF(TRIM(CAST(category AS STRING)), '') IS NOT NULL
    AND NULLIF(TRIM(CAST(particular AS STRING)), '') IS NOT NULL
    AND UPPER(TRIM(CAST(particular AS STRING))) <> 'ALL'

  UNION

  SELECT
    UPPER(TRIM(CAST(category AS STRING))) AS category,
    UPPER(TRIM(CAST(particular AS STRING))) AS particular
  FROM pondolens.dev_silver.regional_obligations
  WHERE NULLIF(TRIM(CAST(category AS STRING)), '') IS NOT NULL
    AND NULLIF(TRIM(CAST(particular AS STRING)), '') IS NOT NULL
    AND UPPER(TRIM(CAST(particular AS STRING))) <> 'ALL'
),
id_start AS (
  SELECT 29999 AS starting_id
)
SELECT
  CAST(
    b.starting_id
      + ROW_NUMBER() OVER (ORDER BY p.category, p.particular)
    AS INT
  ) AS particular_id,
  CAST(p.category AS STRING) AS category,
  CAST(p.particular AS STRING) AS particular
FROM distinct_particulars AS p
CROSS JOIN id_start AS b;

## 2. View the dimension table

Display the IDs and category–particular pairs stored in the Gold table.

In [0]:
%sql
SELECT *
FROM pondolens.dev_gold.dim_particular

## 3. Data quality checks

Check for missing values, duplicates, incorrect IDs, formatting issues, and source pairs missing from the dimension.

- **PASS:** No issues found.
- **REVIEW:** More than 0% up to 5% affected.
- **FAIL:** More than 5% affected.
- Checks with no rows to evaluate return **REVIEW**.

Percentages are based on the rows or columns checked. Duplicate checks count every row in a duplicate group.


In [0]:
%sql
-- Read-only report. PASS: 0%; REVIEW: >0% to 5%; FAIL: >5%.
WITH settings AS (
  SELECT 5.0 AS review_max_pct
),
source_pairs AS (
  SELECT
    UPPER(TRIM(CAST(category AS STRING))) AS category,
    UPPER(TRIM(CAST(particular AS STRING))) AS particular
  FROM pondolens.dev_silver.regional_appropriations
  WHERE NULLIF(TRIM(CAST(category AS STRING)), '') IS NOT NULL
    AND NULLIF(TRIM(CAST(particular AS STRING)), '') IS NOT NULL
    AND UPPER(TRIM(CAST(particular AS STRING))) <> 'ALL'

  UNION

  SELECT
    UPPER(TRIM(CAST(category AS STRING))) AS category,
    UPPER(TRIM(CAST(particular AS STRING))) AS particular
  FROM pondolens.dev_silver.regional_obligations
  WHERE NULLIF(TRIM(CAST(category AS STRING)), '') IS NOT NULL
    AND NULLIF(TRIM(CAST(particular AS STRING)), '') IS NOT NULL
    AND UPPER(TRIM(CAST(particular AS STRING))) <> 'ALL'
),
dim AS (
  SELECT *
  FROM pondolens.dev_gold.dim_particular
),
totals AS (
  SELECT
    (SELECT COUNT(*) FROM dim) AS dim_rows,
    (SELECT COUNT(*) FROM source_pairs) AS source_pair_count
),
duplicate_ids AS (
  SELECT particular_id
  FROM dim
  WHERE particular_id IS NOT NULL
  GROUP BY particular_id
  HAVING COUNT(*) > 1
),
duplicate_pairs AS (
  SELECT
    UPPER(TRIM(category)) AS category,
    UPPER(TRIM(particular)) AS particular
  FROM dim
  GROUP BY UPPER(TRIM(category)), UPPER(TRIM(particular))
  HAVING COUNT(*) > 1
),
checks AS (
  SELECT
    1 AS sort_order,
    'Missing IDs or blank category/particular' AS dq_check,
    COUNT(*) AS affected_count,
    'dimension rows' AS count_basis
  FROM dim
  WHERE particular_id IS NULL
     OR NULLIF(TRIM(category), '') IS NULL
     OR NULLIF(TRIM(particular), '') IS NULL

  UNION ALL

  -- Count every row belonging to a duplicate group.
  SELECT
    2,
    'Duplicate particular IDs',
    COUNT(*),
    'dimension rows'
  FROM dim AS d
  LEFT SEMI JOIN duplicate_ids AS x
    ON d.particular_id = x.particular_id

  UNION ALL

  SELECT
    3,
    'Duplicate normalized category-particular pairs',
    COUNT(*),
    'dimension rows'
  FROM dim AS d
  LEFT SEMI JOIN duplicate_pairs AS x
    ON UPPER(TRIM(d.category)) <=> x.category
    AND UPPER(TRIM(d.particular)) <=> x.particular

  UNION ALL

  SELECT
    4,
    'IDs below 30000',
    COUNT(*),
    'dimension rows'
  FROM dim
  WHERE particular_id < 30000

  UNION ALL

  SELECT
    5,
    'Text not uppercase or trimmed',
    COUNT(*),
    'dimension rows'
  FROM dim
  WHERE category <> UPPER(TRIM(category))
     OR particular <> UPPER(TRIM(particular))

  UNION ALL

  SELECT
    6,
    'Excluded ALL rows present',
    COUNT(*),
    'dimension rows'
  FROM dim
  WHERE UPPER(TRIM(particular)) = 'ALL'

  UNION ALL

  SELECT
    7,
    'Source pairs missing from dimension',
    COUNT(*),
    'source pairs'
  FROM source_pairs AS s
  LEFT ANTI JOIN dim AS d
    ON s.category = d.category
    AND s.particular = d.particular

  UNION ALL

  SELECT
    8,
    'Incorrect or missing column types',
    3 - COUNT(*),
    'columns'
  FROM pondolens.information_schema.columns
  WHERE table_schema = 'dev_gold'
    AND table_name = 'dim_particular'
    AND (
      (column_name = 'particular_id' AND UPPER(data_type) = 'INT')
      OR (
        column_name IN ('category', 'particular')
        AND UPPER(data_type) = 'STRING'
      )
    )
),
evaluated AS (
  SELECT
    c.*,
    CASE c.count_basis
      WHEN 'source pairs' THEN t.source_pair_count
      WHEN 'columns' THEN 3
      ELSE t.dim_rows
    END AS total_checked
  FROM checks AS c
  CROSS JOIN totals AS t
),
percentages AS (
  SELECT
    *,
    100.0 * affected_count / NULLIF(total_checked, 0)
      AS affected_pct
  FROM evaluated
)
SELECT
  p.dq_check,
  p.total_checked,
  p.affected_count,
  ROUND(p.affected_pct, 2) AS affected_percentage,
  -- Use the unrounded percentage when assigning status.
  CASE
    WHEN p.total_checked = 0 THEN 'REVIEW'
    WHEN p.affected_count = 0 THEN 'PASS'
    WHEN p.affected_pct <= s.review_max_pct THEN 'REVIEW'
    ELSE 'FAIL'
  END AS status
FROM percentages AS p
CROSS JOIN settings AS s
ORDER BY p.sort_order;

## 4. Data Quality Results

- **54 unique category - particular pairs**, with IDs from **30000 to 30053**.
- All eight quality checks passed.
- No missing IDs, blank fields, duplicate IDs, or duplicate pairs were found.
- All 54 source pairs are present in the dimension.
- Text formatting and column types passed; no `ALL` particulars remain.